# L8 · Notebook 01 — 线性函数逼近 + 半梯度 TD(0)

**对应 PPT**：`L8-VFA.tex` §2 线性 FA / Tsitsiklis-Van Roy 定理

## 教学目标

1. 实现线性表示 $\hat v(s; w) = \phi(s)^\top w$
2. 半梯度 TD(0) 更新：$w \leftarrow w + \alpha (r + \gamma\, \phi(s')^\top w - \phi(s)^\top w)\, \phi(s)$
3. 在 5-state random walk（Sutton-Barto 例 9.1）上 **on-policy**，权重收敛到投影不动点
4. 对比两种特征：粗特征（2 个组）vs 表格特征（identity），看近似误差差别
5. 与 MC 基线对比方差

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

## 1. 环境：5-state random walk

状态 `0..6`：`0` 和 `6` 是终止态；起点 `3`；每步等概率左/右；右终止得 +1，左终止得 0。

真值（解析）：$v^\pi(s) = s/6$ 对 $s\in\{1,\dots,5\}$。

In [ ]:
def step(s):
    s2 = s + (1 if rng.random() < 0.5 else -1)
    if s2 == 6:
        return s2, 1.0, True
    if s2 == 0:
        return s2, 0.0, True
    return s2, 0.0, False

def episode():
    traj = []
    s = 3
    while True:
        s2, r, done = step(s)
        traj.append((s, r, s2, done))
        if done:
            return traj
        s = s2

v_true = np.array([np.nan, 1/6, 2/6, 3/6, 4/6, 5/6, np.nan])
print('v^π(s) for s=1..5:', v_true[1:6])

## 2. 特征族

- **identity** （表格）：$\phi(s) = e_s$，5 维 → 等价于表格 TD
- **coarse** （粗特征）：把 5 个非终止态分成 `{1,2}` `{3,4,5}`，2 维 → 真值不在张成空间，必有逼近误差

In [ ]:
def phi_identity(s):
    e = np.zeros(5)
    if 1 <= s <= 5:
        e[s - 1] = 1.0
    return e

def phi_coarse(s):
    e = np.zeros(2)
    if 1 <= s <= 2:
        e[0] = 1.0
    elif 3 <= s <= 5:
        e[1] = 1.0
    return e

# 投影不动点（粗特征下）：解 min ||Φw - v_π||²
Phi = np.array([phi_coarse(s) for s in range(1, 6)])
v_pi = v_true[1:6]
w_proj = np.linalg.lstsq(Phi, v_pi, rcond=None)[0]
print('粗特征投影不动点 w_proj =', w_proj.round(4))
print('  → 组 {1,2} 估值 =', w_proj[0].round(4), '(真值平均 0.25)')
print('  → 组 {3,4,5} 估值 =', w_proj[1].round(4), '(真值平均 0.5)')

## 3. 半梯度 TD(0)

$$w \leftarrow w + \alpha\,\bigl(r + \gamma\,\phi(s')^\top w - \phi(s)^\top w\bigr)\,\phi(s)$$

**注意：终止态梯度为 0**（$\phi(\text{terminal})=0$ 使 bootstrap 项消失）。

In [ ]:
def semi_grad_td(phi_fn, dim, n_episodes=2000, alpha=0.05, gamma=1.0, seed=0):
    global rng
    rng = np.random.default_rng(seed)
    w = np.zeros(dim)
    history = []
    for ep in range(n_episodes):
        for s, r, s2, done in episode():
            phi_s = phi_fn(s)
            phi_s2 = phi_fn(s2) if not done else np.zeros(dim)
            td = r + gamma * phi_s2 @ w - phi_s @ w
            w = w + alpha * td * phi_s
        if (ep + 1) % 50 == 0:
            v_hat = np.array([phi_fn(s) @ w for s in range(1, 6)])
            history.append((ep + 1, v_hat))
    return w, history

w_id, hist_id = semi_grad_td(phi_identity, 5, n_episodes=2000, alpha=0.05)
w_co, hist_co = semi_grad_td(phi_coarse,   2, n_episodes=2000, alpha=0.05)

v_id_final = np.array([phi_identity(s) @ w_id for s in range(1, 6)])
v_co_final = np.array([phi_coarse(s)   @ w_co for s in range(1, 6)])
print('表格特征  v_hat =', v_id_final.round(3))
print('粗特征    v_hat =', v_co_final.round(3))
print('真值      v_π   =', v_pi.round(3))
print('粗特征投影 v_proj=', (Phi @ w_proj).round(3))

## 4. 收敛曲线：每 50 episodes 记录 $\hat v$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for hist, ax, title, vproj in [
    (hist_id, axes[0], 'Tabular (identity features)', v_pi),
    (hist_co, axes[1], 'Coarse 2-group features',      Phi @ w_proj),
]:
    eps = [h[0] for h in hist]
    arr = np.array([h[1] for h in hist])
    for s_idx in range(5):
        ax.plot(eps, arr[:, s_idx], label=f's={s_idx+1}')
    for s_idx in range(5):
        ax.axhline(vproj[s_idx], color=f'C{s_idx}', linestyle='--', alpha=0.3)
    ax.set_xlabel('episode'); ax.set_ylabel(r'$\hat v(s)$')
    ax.set_title(title)
    ax.legend(fontsize=8, ncol=2)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/linear_td_convergence.png', dpi=120, bbox_inches='tight')
plt.show()
print('虚线 = 投影不动点。表格特征 → 收敛到真值；粗特征 → 收敛到投影（必有近似误差）。')

## 5. 课堂诊断小结

| 论断 | 数值证据 |
|---|---|
| **on-policy 线性 TD 收敛**（Tsitsiklis-Van Roy 1997）| 两种特征均稳定收敛 |
| 收敛点 = **投影不动点** $\Phi w^* = \Pi_\mu T^\pi \Phi w^*$ | 粗特征 $\hat v$ 重合在分组平均处 |
| 不是真值，但 $\|\Phi w^* - v^\pi\|_\mu \le \tfrac{1}{1-\gamma}\|\Pi v^\pi - v^\pi\|_\mu$ | 见 PPT line 184-209 定理 |

## 思考题

1. 为什么叫**半**梯度？真梯度会多算哪一项？为什么去掉它仍然收敛？
2. 把行为策略改成右多左少（off-policy），再跑一次，会发生什么？（→ Notebook 02 Baird's）
3. 粗特征下增大 episode 数能消除逼近误差吗？为什么不能？